# HEAD-QA (dvilares/head_qa) — Data Inspection

Spanish healthcare professional MCQ (medicine, nursing, psychology,
chemistry, pharmacology, biology).

Key concerns:
- ~14% of medicine questions require an image to answer — must filter these
- 2013-2014 exams have 5 options; 2015+ have 4 — both are fine, note distribution
- ra field is 1-indexed int matching aid — need to map to letter
- Produces cleaned CSV if image-dependent questions are common

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

# Load Spanish config — has train/validation/test splits
ds = load_dataset('dvilares/head_qa', 'es')
print(ds)

In [ ]:
# Inspect raw structure
ex = ds['train'][0]
print('Keys:', list(ex.keys()))
print()
for k, v in ex.items():
    if k != 'image':
        print(f'{k}: {repr(v)[:200]}')
print(f'image: {type(ex["image"])} (is None: {ex["image"] is None})')

In [ ]:
# Flatten all splits into a dataframe
rows = []
for split in ['train', 'validation', 'test']:
    for ex in ds[split]:
        answers = ex['answers']  # list of dicts: {aid, atext}
        ra = int(ex['ra'])       # 1-indexed correct answer id
        has_image = ex['image'] is not None
        
        # Find the correct answer text
        correct_atext = next(
            (a['atext'] for a in answers if a['aid'] == ra), None
        )
        
        rows.append({
            'split':         split,
            'qid':           ex['qid'],
            'name':          ex['name'],
            'year':          ex['year'],
            'category':      ex['category'],
            'qtext':         ex['qtext'],
            'n_answers':     len(answers),
            'ra':            ra,
            'correct_atext': correct_atext,
            'has_image':     has_image,
            'aids':          [a['aid'] for a in answers],
            'atexts':        [a['atext'] for a in answers],
        })

df = pd.DataFrame(rows)
print(df.shape)
df.head(2)

In [ ]:
# Split sizes
print(df.groupby('split').size())

In [ ]:
# Category distribution
print(df.groupby(['split', 'category']).size().to_string())

In [ ]:
# Answer count distribution (4 vs 5 options)
print('Answer count distribution by year:')
print(df.groupby(['year', 'n_answers']).size().to_string())
print()
print('Answer count distribution overall:')
print(df['n_answers'].value_counts())

In [ ]:
# Image question analysis
print(f'Total rows with image: {df["has_image"].sum()} / {len(df)}')
print()
print('Image questions by category:')
print(df.groupby(['category', 'has_image']).size().to_string())

In [ ]:
# Inspect image question text — do they explicitly reference an image?
# If the question says "see figure"/"imagen" etc, the image is required
image_rows = df[df['has_image']]
print(f'Sample image question texts:')
for _, row in image_rows.head(10).iterrows():
    print(f'  [{row["category"]}] {row["qtext"][:120]}')
    print()

In [ ]:
# Check if image questions are answerable without the image
# Heuristic: if question text contains reference to image/figure, likely not answerable
image_ref_pattern = re.compile(
    r'\b(imagen|figura|radiograf|rx|\bfoto\b|\bfig\.?\b|image|figure|radiograph)',
    re.IGNORECASE
)

df['image_ref_in_text'] = df['qtext'].apply(
    lambda x: bool(image_ref_pattern.search(str(x)))
)

print('Rows with explicit image reference in question text:')
print(df.groupby(['has_image', 'image_ref_in_text']).size())
print()
# False positives (no image but mentions image) and 
# true positives (has image AND mentions image)
must_filter = df[df['has_image'] & df['image_ref_in_text']]
maybe_ok = df[df['has_image'] & ~df['image_ref_in_text']]
print(f'Has image + mentions image (must filter): {len(must_filter)}')
print(f'Has image + no image ref (maybe answerable): {len(maybe_ok)}')

In [ ]:
# Spot-check "maybe ok" image questions
print('Sample image questions with no explicit image reference in text:')
for _, row in maybe_ok.head(5).iterrows():
    print(f'  [{row["category"]}] {row["qtext"][:150]}')
    print()

In [ ]:
# Check for ra not matching any aid
bad_ra = df[df['correct_atext'].isna()]
print(f'Rows where ra does not match any aid: {len(bad_ra)}')
if len(bad_ra):
    display(bad_ra[['qid', 'category', 'qtext', 'ra', 'aids']].head(10))

In [ ]:
# Encoding corruption check
corruption = re.compile(r'[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]')

def has_corruption(row):
    fields = [row['qtext']] + list(row['atexts'])
    return any(corruption.search(str(f)) for f in fields)

df['has_corruption'] = df.apply(has_corruption, axis=1)
print(f'Rows with encoding corruption: {df["has_corruption"].sum()}')

In [ ]:
# Spot-check formatted examples per category
for cat in df['category'].unique():
    row = df[(df['category'] == cat) & ~df['has_image']].iloc[0]
    print(f'=== {cat} ({row["year"]}) ===')
    print(f'Q: {row["qtext"]}')
    letters = 'ABCDE'
    for i, (aid, atext) in enumerate(zip(row['aids'], row['atexts'])):
        marker = ' ← correct' if aid == row['ra'] else ''
        print(f'  {letters[i]}) {atext}{marker}')
    print()

In [ ]:
# Cleaning decision
# Strategy: filter out image questions entirely (conservative)
# Alternative: only filter rows where image_ref_in_text=True
# Using conservative strategy for SFT data quality

train_val = df[df['split'].isin(['train', 'validation'])].copy()

clean = train_val[
    ~train_val['has_image'] &       # no image questions
    train_val['correct_atext'].notna() &  # valid answer
    ~train_val['has_corruption'] &
    (train_val['qtext'].str.strip() != '')
].copy()

rejected = train_val[~train_val.index.isin(clean.index)].copy()

print(f'Train+val total: {len(train_val)}')
print(f'Clean:           {len(clean)}')
print(f'Rejected:        {len(rejected)}')
print()
print('Clean by category:')
print(clean['category'].value_counts().to_string())
print()
print('Clean by n_answers:')
print(clean['n_answers'].value_counts())

In [ ]:
# Save cleaned version
import json
from pathlib import Path

out_dir = Path('multilingual/data/processed/head_qa')
out_dir.mkdir(parents=True, exist_ok=True)

for col in ['aids', 'atexts']:
    clean[col] = clean[col].apply(json.dumps)
    rejected[col] = rejected[col].apply(json.dumps)

clean.to_csv(out_dir / 'head_qa_clean.csv', index=False)
rejected.to_csv(out_dir / 'head_qa_rejected.csv', index=False)

print(f'Saved: {out_dir / "head_qa_clean.csv"} ({len(clean)} rows)')
print(f'Saved: {out_dir / "head_qa_rejected.csv"} ({len(rejected)} rows)')

In [ ]:
# Summary
print('=== SUMMARY ===')
print(f'Total rows (train+val): {len(train_val)}')
print(f'Image questions filtered: {train_val["has_image"].sum()}')
print(f'Clean rows: {len(clean)}')
print()
print('Answer count in clean data:')
print(clean['n_answers'].value_counts().to_string())
print()
print('Notes:')
print('  - ra is 1-indexed int; loader must map to letter via position')
print('  - aids are not always 1,2,3,4 — use position index, not aid value')
print('  - 5-option questions acceptable (benchmark allows 2-20 choices)')
print('  - conservative: all image questions filtered regardless of text content')